In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from scipy.optimize import linear_sum_assignment

In [ ]:
df = pd.read_csv("wildchat_sample_scored_clean.csv", on_bad_lines='skip')
df = df[df['empathy_score'] != 4].copy() #take out fours
df['treatment_group'] = np.where(df['empathy_score'] >= 5, 1, 0) #is 0 if not 5 in Sempathy
df = df[df['user_prompt'].notna()].copy()
df['user_prompt'] = df['user_prompt'].astype(str).str.strip()
df = df.drop_duplicates(subset=['user_prompt']).copy()


treat_df = df[df['treatment_group'] == 1].reset_index(drop=True)
ctrl_df = df[df['treatment_group'] == 0].reset_index(drop=True)



In [6]:
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
treat_emb = model.encode(treat_df['user_prompt'].tolist(), normalize_embeddings=True)
ctrl_emb = model.encode(ctrl_df['user_prompt'].tolist(), normalize_embeddings=True)


In [8]:
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sentence_transformers import util

# Step 1: Compute cosine similarity matrix
similarity = util.cos_sim(treat_emb, ctrl_emb).cpu().numpy()  # shape: (num_treat, num_ctrl)

# Step 2: Use Hungarian algorithm to maximize total similarity
# Hungarian algorithm *minimizes* cost, so we negate the similarity matrix
row_ind, col_ind = linear_sum_assignment(-similarity)

# Step 3: Collect matched pairs
matched_pairs = []
for i, j in zip(row_ind, col_ind):
    matched_pairs.append({
        'treatment_id': treat_df.index[i],
        'control_id': ctrl_df.index[j],
        'similarity': similarity[i][j],
        'treatment_prompt': treat_df.loc[i, 'user_prompt'],
        'control_prompt': ctrl_df.loc[j, 'user_prompt']
    })

matched_df = pd.DataFrame(matched_pairs)

# Step 4: Merge back metadata (optional)
matched_df = matched_df.merge(
    df[['user_prompt', 'llm_response', 'empathy_score']],
    left_on='treatment_prompt',
    right_on='user_prompt',
    how='left',
    suffixes=('', '_treat')
).merge(
    df[['user_prompt', 'llm_response', 'empathy_score']],
    left_on='control_prompt',
    right_on='user_prompt',
    how='left',
    suffixes=('', '_control')
).drop(columns=['user_prompt', 'user_prompt_control'])

print(matched_df.head())

matched_df = pd.DataFrame(matched_pairs)

# Optional: merge back LLM answers or empathy scores if you need them
matched_df = matched_df.merge(
    df[['user_prompt', 'llm_response', 'empathy_score']],
    left_on='treatment_prompt',
    right_on='user_prompt',
    how='left',
    suffixes=('', '_treat')
).merge(
    df[['user_prompt', 'llm_response', 'empathy_score']],
    left_on='control_prompt',
    right_on='user_prompt',
    how='left',
    suffixes=('', '_control')
).drop(columns=['user_prompt', 'user_prompt_control'])



   treatment_id  control_id  similarity  \
0             0        1403    0.290373   
1             1        1593    0.451548   
2             2        1841    0.349750   
3             3        1768    0.396039   
4             4         923    0.543507   

                                    treatment_prompt  \
0                         Maybe i need a little help   
1                            let's make some scripts   
2  can you do a bizarre day hitbox expander scrip...   
3  Ha, I have found a seam. Hero is actually a gu...   
4  # make a prompt for making an AI to act as a s...   

                                      control_prompt  \
0                                   are you sentient   
1  now, these lines of codes do i put them into a...   
2  You can generate a base10 colour image with a ...   
3        Tell me about Margot relationship with men?   
4  if i want to use AI to generate texts that wil...   

                                        llm_response  empathy_score

In [9]:
matched_df.to_csv("matched_df.csv", index=False)